In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:98% !important;}
div.cell.code_cell.rendered{width:98%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li, div.text_cell_render p{width:95% !important;font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))

## RAG 절차
- 1. docx문서를 읽는다(python-docx) 
- 2. 읽어온 문서를 쪼갠다(text_spliter이용)
    * 모델의 context window와 비용 확인
    * chunk(문서를 쪼갠 하나)가 길면 비용과 시간이 많이 소요
- 3. 쪼갠 문서를 임베딩 -> vector Store에 저장 => chroma(local 파일)
- 4. 질문과 vector DB의 유사도 검색
- 5. 유사도검색으로 가져온 문서를 LLM에 질문과 같이 전달하여 답변 생성(LangChain)

# 0. 패키지 설치

In [ ]:
# 문서읽어오기
#%pip install -q docx2txt

In [ ]:
# 텍스트를 chunk로 나누는 기능 
%pip install -q langchain-text-splitters

In [ ]:
# 파일 저장 chroma (cf. chromadb와 다름)
%pip install -q langchain-chroma

In [ ]:
# langchain 사용
%pip install -q langchain

# 1. 문서읽기(x)

In [ ]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
loader = Docx2txtLoader('./tax_docx/소득세법(법률)(제21221호)(20260701).docx')
document = loader.load()
len(document)

# 2. 문서를 쪼개면서 읽기(o)
## 2.1 1500토큰씩 겹쳐서 쪼개면서 읽기

In [ ]:
import time
start = time.time()
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import TokenTextSplitter
loader = Docx2txtLoader('./tax_docx/소득세법(법률)(제21221호)(20260701).docx')
# gpt-4o-mini, .... 모델의 토큰 추출 방식 : o200k_base방식
# gpt-4,  gpt4 turbo .. 모델의 토큰 추출방식 : cl100k_base방식
text_splitter = TokenTextSplitter(
        encoding_name="cl100k_base", # 토큰을 세는 방식 이름
        chunk_size=1500,             # 1500토큰 수 기준으로 chunk 분리
        chunk_overlap=200,
        # separator 파라미터가 없어
)
documents = loader.load_and_split(text_splitter=text_splitter)
runtime = time.time() - start
print('문서를 겹쳐 쪼개면서 읽어오는 시간 :', runtime)
print('chunk갯수 :', len(documents))

In [ ]:
# 각 chunk별 글자수 : len(documents[0].page_content)
print([len(document.page_content) for document in documents] )

In [ ]:
# chunk 평균 글자수, 최소 글자수, 최대 글자수
import numpy as np
print(np.array([len(document.page_content) for document in documents]).mean())
print(max([len(document.page_content) for document in documents]))
print(min([len(document.page_content) for document in documents]))

## 2.2 1500글짜씩 겹쳐 쪼개 읽어오기

In [3]:
import time
start = time.time()
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader("tax_docx/소득세법(법률)(제21221호)(20260701).docx")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,   # 문서를 쪼갤 때 1500글자씩 chunking
    chunk_overlap=20, # 20문자씩 overlap 하여 chunking
    # separators=["\n\n", "\n", " ", ""]
)
# Recursive(재귀적으로) 다음의 순서대로 chunking 시도 : "\n\n", "\n", " ", ""
documents = loader.load_and_split(text_splitter=text_splitter)
runtime = time.time() - start
print('문서를 1500글자로 겹쳐 쪼개면서 읽는 시간 :', runtime)
print('chunk갯수 :', len(documents))

문서를 1500글자로 겹쳐 쪼개면서 읽는 시간 : 3.7908198833465576
chunk갯수 : 180


In [4]:
# chunk들의 글자수
print([len(document.page_content) for document in documents])

[1446, 1421, 1444, 1486, 1188, 1330, 1456, 1257, 1472, 1316, 1347, 1447, 1458, 1343, 1485, 1333, 1478, 1439, 1419, 1445, 1496, 1470, 1455, 1483, 1487, 1382, 1207, 1467, 1399, 1484, 1327, 1420, 1383, 1431, 1387, 1002, 1495, 1484, 1473, 934, 1460, 1471, 1348, 1320, 1285, 1460, 1387, 1478, 1498, 1352, 1418, 1432, 1372, 986, 1441, 1495, 1465, 1439, 1363, 1433, 1475, 1459, 1371, 1398, 1477, 1318, 1417, 1495, 1463, 1443, 1472, 1276, 1391, 1283, 1456, 1354, 1426, 1226, 1492, 1489, 1295, 1483, 1439, 1304, 1451, 1490, 1318, 1364, 1479, 931, 1401, 1478, 1308, 1369, 1491, 1468, 1453, 1414, 1489, 1360, 1458, 1427, 1491, 1495, 1478, 1391, 1416, 1412, 1373, 1349, 1445, 1485, 1367, 1384, 1383, 1452, 1453, 1376, 1419, 1457, 1444, 1419, 1457, 1312, 1467, 1477, 1419, 1475, 1331, 1492, 1491, 1461, 1486, 1479, 1475, 1385, 1305, 1486, 1290, 1492, 1412, 1387, 1495, 1250, 1154, 1330, 1310, 1385, 1462, 1455, 1449, 1365, 1494, 1360, 1426, 1422, 1452, 1388, 1343, 1325, 1467, 1455, 1399, 1375, 1469, 1459, 1138, 

In [ ]:
documents[2]

# 3. 쪼갠 문서를 임베딩 -> 벡터 데이터 베이스(파일)에 저장
- 임베딩 모델 : upstage의 embedding-query, solar-embedding-1-large-passage
- vector DB 저장 폴더 : ./chroma

In [28]:
# 임베딩 객체
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()
embeding = UpstageEmbeddings(
           # api_key=os.getenv('UPSTAGE_API_KEY'),
           model = 'solar-embedding-1-large-passage'
            )

In [29]:
# embed_query() : 한 문자열을 임베딩 벡터로 전환
len(embeding.embed_query("소"))

4096

In [7]:
# embed_documents() : 여러 문자열을 임베딩 벡터로 전환
vectors = embeding.embed_documents([
    documents[0].page_content, documents[1].page_content
])

In [8]:
print(len(vectors[0]), len(vectors[1]))
print(vectors[0][:10])

4096 4096
[0.0100555419921875, 0.00269317626953125, -0.016448974609375, 0.00870513916015625, -0.017181396484375, -0.0040283203125, -0.00180816650390625, 0.0004897117614746094, -0.0148468017578125, -0.001678466796875]


In [11]:
%%time
from langchain_chroma import Chroma
# 데이터를 처음 저장할 때
# database = Chroma.from_documents(
#     documents=documents, # chunking 작업한 documents
#     embedding=embeding, # 임베딩 객체
#     collection_name='tax-collection', # 생략시 이름이 랜덤
#     persist_directory='./chroma_upstage'      # 생략시 로컬에 저장 안 됨. 인메모리 vector DB
# )
# 저장된 vector DB를 가져올 때
database = Chroma(
    embedding_function=embeding,
    collection_name='tax-collection',
    persist_directory='./chroma_upstage'
)

CPU times: total: 15.6 ms
Wall time: 13.4 ms


In [ ]:
# 확인
results = database.get(include=['embeddings', 'documents', 'metadatas'])
print('데이터수 :', len(results['ids']))
print('문서 임베딩 차원 수 :', len(results['embeddings'][0]))
print('1번째 원본 :', results['documents'][1][:100])
print('1번째 원본의 임베딩 벡터 :', results['embeddings'][1][:10])
print('1번째 metadata :', results['metadatas'][1])

# 4. vector DB에 질문과 유사도 검색(retrieval)

In [12]:
query = "연봉 5천만원인 직장인의 소득세는 얼마예요?"
retrieved_docs = database.similarity_search(query=query,
                                           k=2) # 기본 k값은 4

In [13]:
len(retrieved_docs)

2

In [ ]:
l = ['무시기무시기', '어쩌구저쩌구']
print("\n--\n".join(l))

In [16]:
# 검색된 문서를 LLM에 전달하기 위한 가공
retrieved_doc = "\n--\n".join([doc.page_content.replace('\n\n', ' ') for doc in retrieved_docs])

# 5. 유사도 검색으로 가져온 문서를 질문과 같이 LLM에 전달하여 답변생성-1

In [17]:
from langchain_openai import ChatOpenAI
load_dotenv()
llm = ChatOpenAI(model='gpt-4.1-nano')

In [18]:
from langchain_upstage import ChatUpstage
load_dotenv()
llm = ChatUpstage(model='solar-pro4')

In [19]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세법 전문가입니다
- [context]를 참조해서 사용자의 질문에 답변해 주세요. context는 다음과 같습니다
{retrieved_doc}
질문 : {query}"""
ai_message = llm.invoke(prompt) # 과금

In [20]:
ai_message.usage_metadata # 사용된 토큰량

{'input_tokens': 1636,
 'output_tokens': 785,
 'total_tokens': 2421,
 'input_token_details': {'audio': 0, 'cache_read': 0},
 'output_token_details': {'audio': 0, 'reasoning': 0}}

In [21]:
print(ai_message.content)

제공해주신 소득세법 조항과 현행 한국 소득세 구조를 바탕으로 설명드리면, **연봉 5천만 원인 직장인의 최종 소득세는 단순히 하나의 금액으로 산출되지 않습니다.** 소득세는 연봉(총급여액)이 아니라 **각종 소득공제와 세액공제를 거쳐 결정된 종합소득과세표준에 세율을 곱한 후, 다시 세액공제를 반영**하는 과정을 거치기 때문입니다.

아래는 제공하신 조문과 일반적인 직장인의 기본 공제 조건(부양가족 1명, 특별공제 없음 등)을 가정한 **참고용 예시 계산 흐름**입니다.

### 1. 근로소득공제 (제59조 관련 전후 규정 및 표준 공제 구조)
* 총급여액: 5,000만 원
* 근로소득공제(표준 구간 적용): 약 750만 원
* 근로소득금액: 5,000만 원 - 750만 원 = **4,250만 원**

### 2. 종합소득과세표준 및 산출세액
* 종합소득과세표준: 근로소득금액 약 4,250만 원 (기본공제 150만 원 등 인적공제 적용 시 약 4,100만 원 전후)
* 기본세율(6%~45% 누진세율) 적용 시: 과세표준이 1,400만 원을 초과~5,000만 원 이하 구간에 해당하므로, **산출세액 약 400만 원~600만 원 수준**이 됩니다.(정확한 누진공제액 반영 시 개인 조건 따라 차이)

### 3. 세액공제 반영 (제공 조문 활용)
제공해주신 조문에 따라 다음과 같은 공제를 산출세액에서 차감합니다.
* **근로소득세액공제(제59조)**: 총급여 5,000만 원은 제59조 제2항 제2호(3,300만 원 초과~7,000만 원 이하)에 해당. 계산식 및 단서 적용 시 공제세액은 **66만 원**(66만 원 미만일 수 없는 하한 적용)으로 볼 수 있습니다.
* **자녀세액공제(제59조의2)**: 기본공제대상 자녀가 1명일 경우 연 25만 원, 2명일 경우 연 55만 원, 3명 이상일 경우 55만 원에 2명 초과 1명당 40만 원을 합산한 금액.
* 그 외 보험료·교육비·의료비 등 **특별세액공제**나, 해당 연도에 재해손실이 있을 경우 **재해손실세액공제(제5

# 5. 유사도 검색으로 가져온 문서를 질문과 같이 LLM에 전달하여 답변생성-2

In [22]:
from langchain_core.prompts import ChatPromptTemplate
promptTemplate = ChatPromptTemplate([
    ("system", "당신은 최고의 한국 소득세법 전문가입니다."),
    ("human", """다음 문맥을 참고하여 질문에 답변하세요. 답을 모르면 모른다고 답변하세요.
    최대 3문장으로 간결하게 답변하세요.
    질문 : {question}
    문맥 : {context}
    답변 :""")
])
promptTemplate

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 최고의 한국 소득세법 전문가입니다.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='다음 문맥을 참고하여 질문에 답변하세요. 답을 모르면 모른다고 답변하세요.\n    최대 3문장으로 간결하게 답변하세요.\n    질문 : {question}\n    문맥 : {context}\n    답변 :'), additional_kwargs={})])

In [23]:
prompt = promptTemplate.invoke({'context':retrieved_doc, # retrieved_docs 비추
                                'question':query})

In [24]:
ai_message = llm.invoke(prompt) # 과금

In [25]:
from langchain_core.output_parsers import StrOutputParser
outputParser = StrOutputParser()
outputParser.invoke(ai_message)

'제시된 문맥은 소득세법 상의 각종 세액공제 및 소득공제 규정을 담고 있을 뿐, 최종 소득세액을 계산할 수 있는 종합소득세율표나 구체적인 산출 방식이 포함되어 있지 않습니다. 따라서 제공된 정보만으로는 연봉 5천만원인 직장인의 정확한 소득세액을 특정하여 답변드릴 수 없습니다.'

In [26]:
# 위의 작업을 한번에 
outputParser.invoke(llm.invoke(promptTemplate.invoke(
    {'context' : retrieved_doc, 'question':query}
)))

'제시된 문맥에는 근로소득세액공제의 한도액과 계산 방식만 명시되어 있어, 연봉 5천만 원의 종합소득산출세액 및 각종 공제를 거쳐 실제 납부해야 할 소득세를 정확히 산정할 수 없으므로 알 수 없습니다. 소득세액은 과세표준 구간, 기본공제, 원천징수 방식 등에 따라 크게 달라지므로, 정확한 금액은 국세청 홈택스 모의계산을 이용하거나 세무 전문가에게 상담받으시기 바랍니다.'

# 6. langchain으로 답변 생성

In [27]:
rag_chain = promptTemplate | llm | outputParser
rag_chain.invoke({'context':retrieved_doc, 'question':query})

'제공된 문맥에는 근로소득공제나 기본공제 등에 대한 규정만 있을 뿐, 연봉을 바탕으로 실제 소득세를 계산하는 구체적인 세율표가 포함되어 있지 않아 정확한 소득세를 알 수 없습니다. 소득세액은 총급여에서 각종 공제를 차감한 과세표준에 기본세율(6~45%)을 적용하고 근로소득세액공제 등을 적용해야 산출되므로, 주어진 내용만으로는 금액을 특정할 수 없습니다.'

Langchain으로 전달

In [31]:
# from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_upstage import ChatUpstage, UpstageEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
# 1. 임베딩 & LLM 초기화
load_dotenv()
# embedding = OpenAIEmbeddings(model = "text-embedding-3-large")
# llm = ChatOpenAI(model='gpt-4.1-mini')
embedding = UpstageEmbeddings(model='solar-embedding-1-large-passage')
llm = ChatUpstage(model='solar-pro4')
# 2. vector DB load
vectorStore = Chroma(
    embedding_function=embedding,
    persist_directory='./chroma_upstage/',
    collection_name='tax-collection'
)
# 3. Retriever 생성
# (1) search_type='similarity' : 가장 유사한 top-k개 추출
retriever = vectorStore.as_retriever(search_type='similarity', 
                                     search_kwargs={'k':3}) # 유사도가 높은 top-3추출
# # (2) search_type='mmr' : 후보 10개 중 서로 덜 겹치는 3개 추출
# retriever = vectorStore.as_retriever(search_type='mmr', 
#                                      search_kwargs={'k':3,
#                                                    'fetch_k':10,
#                                                    'lambda_mult':0.5}) # 1에 가까울수록 관련성 깊은 계산
# # (3) search_type='similarity_score_threshold' : 
# retriever = vectorStore.as_retriever(search_type='similarity_score_threshold', 
#                                      search_kwargs={'k':3, 'score_threshold':0.4}) 
# 4. 프롬프트템플릿
template = f"""당신은 최고의 한국 소득세법 전문가입니다. 다음 문맥을 참고하여 질문에 답하세요. 답을 모르면 모른다고 
답하세요. 최대 3문장으로 간결하게 답변하세요
질문 : {{query}}
문맥 : {{context}}
답변 :"""
prompt = ChatPromptTemplate.from_template(template)
# 5. 검색된 문서를 텍스트로 변환하는 함수
def format_documents(retrieved_docs):
    return "\n--\n".join([doc.page_content.replace('\n\n', ' ') for doc in retrieved_docs])

In [32]:
# 6. RAG 체인 구성
rag_chain = (
    {
        "context":retriever | format_documents,
        "query":RunnablePassthrough() # 질문만 그대로 전달
    }
    | prompt
    | llm
    | StrOutputParser()
)
rag_chain.invoke('연봉 5천만원인 직장인의 소득세는 얼마인가요?')

'제공된 문맥에는 근로소득공제 계산 방식이나 종합소득세율 등 소득세 산출에 필요한 핵심 정보가 포함되어 있지 않아, 연봉 5천만원에 대한 정확한 소득세 금액을 산출할 수 없습니다. 소득세액은 근로소득공제, 각종 인적·특별세액공제 등 개인별 공제 요건을 반영해야 하므로, 정확한 금액은 관할 세무서 또는 세무 전문가에게 문의하셔야 합니다.'